In [1]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

sys.path.append("..")
from src import create_sentence_nace_code_similarities, analysis_functions
import test_base
from sentence_splitter import split_text_into_sentences

## Test retrieving the similarities for chunks in a pdf to the NACE Code

**Function:** pdf-> (chunk x code -> [-1,1])

**Parameters:** 

- pdf_path
- way of chunking the text (e.g. sentences, sliding window, or paragraphs)
- way of preprocessing (most is fixed for all reports)
    - similarity threshold of relevant chunks
    - length of irrelevant chunks

**Store analytics for each datapoint:**

- mean score for each class given a threshold

In [2]:
# Parameters: 

threshold_min_chunk_len = 100
cos_threshold = 0.4
sentence_length = 6

In [3]:
dataset_path = "../data/datasets/german_annual_reports"
dataset_path = "../data/datasets/stoxx_600"
dataset_path = "../data/datasets/stoxx_600_extended"
dataset_path = "../data/datasets/reports_subset_from_full_data_1"

In [4]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [5]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0)
nace_classes.head()

,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,NACE,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,1.19,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,1.41,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,2.30,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,1.19,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,1.19,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf


In [6]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

{'Ariston Holdings Ltd.1.pdf': 1.19,
 'Heritage Foods Limited1.pdf': 1.41,
 'Timberwell Bhd.1.pdf': 2.3,
 'Matang Bhd.2.pdf': 1.19,
 'Sin Heng Chan (Malaya) Bhd.1.pdf': 1.19,
 'Tech-bank Food Co., Ltd.3.pdf': 1.46,
 'Namoi Cotton Ltd1.pdf': 1.63,
 'Atlantic Sapphire ASA1.pdf': 3.11,
 'Agra Limited2.pdf': 1.19,
 'Huisheng International Holdings Ltd.3.pdf': 1.46,
 'Australian Agricultural Company Limited1.pdf': 1.62,
 'Waterbase Limited2.pdf': 3.21,
 'CannAmerica Brands Corp.1.pdf': 1.3,
 'Qian Hu Corporation Limited1.pdf': 3.21,
 'Jawala Inc.1.pdf': 1.19,
 'Green Thumb Industries Inc.1.pdf': 1.19,
 'CLS Holdings USA Inc2.pdf': 1.19,
 'China Bozza Development Holdings Limited1.pdf': 2.4,
 'Salmon Evolution ASA1.pdf': 3.21,
 'North American Cannabis Holdings, Inc.1.pdf': 1.19,
 'Malwatte Valley Plantations Plc1.pdf': 1.61,
 'Greenheart Group Limited1.pdf': 2.2,
 'Bumitama Agri Ltd.1.pdf': 1.19,
 'Genus plc1.pdf': 1.62,
 'Kotagala Plantations Plc1.pdf': 2.3,
 'PT Andira Agro Tbk1.pdf': 1.1

In [7]:
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
report_to_nace_class

{'Ariston Holdings Ltd.1.txt': 1.19,
 'Heritage Foods Limited1.txt': 1.41,
 'Timberwell Bhd.1.txt': 2.3,
 'Matang Bhd.2.txt': 1.19,
 'Sin Heng Chan (Malaya) Bhd.1.txt': 1.19,
 'Tech-bank Food Co., Ltd.3.txt': 1.46,
 'Namoi Cotton Ltd1.txt': 1.63,
 'Atlantic Sapphire ASA1.txt': 3.11,
 'Agra Limited2.txt': 1.19,
 'Huisheng International Holdings Ltd.3.txt': 1.46,
 'Australian Agricultural Company Limited1.txt': 1.62,
 'Waterbase Limited2.txt': 3.21,
 'CannAmerica Brands Corp.1.txt': 1.3,
 'Qian Hu Corporation Limited1.txt': 3.21,
 'Jawala Inc.1.txt': 1.19,
 'Green Thumb Industries Inc.1.txt': 1.19,
 'CLS Holdings USA Inc2.txt': 1.19,
 'China Bozza Development Holdings Limited1.txt': 2.4,
 'Salmon Evolution ASA1.txt': 3.21,
 'North American Cannabis Holdings, Inc.1.txt': 1.19,
 'Malwatte Valley Plantations Plc1.txt': 1.61,
 'Greenheart Group Limited1.txt': 2.2,
 'Bumitama Agri Ltd.1.txt': 1.19,
 'Genus plc1.txt': 1.62,
 'Kotagala Plantations Plc1.txt': 2.3,
 'PT Andira Agro Tbk1.txt': 1.1

In [8]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
reports_path

['../data/datasets/reports_subset_from_full_data_1/TXTs/PVH Corp.3.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Mekdam Holding Group Company1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Ambea AB1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Harbour Equine Holdings Limited3.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Poste Italiane SpA1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Propel Funeral Partners Ltd.1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Nabors Industries Ltd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/MK Land Holdings Bhd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Brookfield Infrastructure Corp. (New York)2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Shangri-La Hotel Public Co. Ltd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Rentokil Initial plc1.txt',
 '../data/datasets/reports_subset_fro

In [9]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [10]:
for i in range(4,5): 
    nace_level = i

    result_path = f"../results/dataset__{dataset_name}_sentence_len_{sentence_length}__min_chunk_len_{threshold_min_chunk_len}__cos_thresh_{cos_threshold}__nace_level_{nace_level}"
    #print("Store at: ", result_path)

    res = test_base.test_similarities(reports_path, preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i, overwrite=True)

  0%|          | 0/1555 [00:00<?, ?it/s]

{'P_EDUCATION': 0.015000000000000001, 'K_FINANCIAL AND INSURANCE ACTIVITIES': 0.014833333333333334, 'L_REAL ESTATE ACTIVITIES': 0.00675, 'M_PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIVITIES': 0.005631578947368421, 'N_ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES': 0.005515151515151515, 'O_PUBLIC ADMINISTRATION AND DEFENCE; COMPULSORY SOCIAL SECURITY': 0.004222222222222222, 'I_ACCOMMODATION AND FOOD SERVICE ACTIVITIES': 0.003875, 'G_WHOLESALE AND RETAIL TRADE; REPAIR OF MOTOR VEHICLES AND MOTORCYCLES': 0.002879120879120879, 'J_INFORMATION AND COMMUNICATION': 0.0025, 'Q_HUMAN HEALTH AND SOCIAL WORK ACTIVITIES': 0.002416666666666667, 'U_ACTIVITIES OF EXTRATERRITORIAL ORGANISATIONS AND BODIES': 0.002, 'T_ACTIVITIES OF HOUSEHOLDS AS EMPLOYERS; UNDIFFERENTIATED GOODS- AND SERVICES-PRODUCING ACTIVITIES OF HOUSEHOLDS FOR OWN USE': 0.0013333333333333333, 'S_OTHER SERVICE ACTIVITIES': 0.0012631578947368421, 'F_CONSTRUCTION': 0.0009090909090909091, 'H_TRANSPORTATION AND STORAGE': 0.00073913043478

 20%|██        | 317/1555 [01:33<06:03,  3.41it/s]

{'64.3_Trusts, funds and similar financial entities': 0.029, '85.6_Educational support activities': 0.029, '64.2_Activities of holding companies': 0.026, '66.3_Fund management activities': 0.025, '64.9_Other financial service activities, except insurance and pension funding': 0.024999999999999998, '69.2_Accounting, bookkeeping and auditing activities; tax consultancy': 0.024, '65.3_Pension funding': 0.023, '85.4_Higher education': 0.0225, '70.2_Management consultancy activities': 0.022, '77.4_Leasing of intellectual property and similar products, except copyrighted works': 0.019, '85.3_Secondary education': 0.0175, '01.5_Mixed farming': 0.016, '84.3_Compulsory social security activities': 0.014, '66.1_Activities auxiliary to financial services, except insurance and pension funding': 0.013666666666666667, '82.1_Office administrative and support activities': 0.0135, '85.5_Other education': 0.012750000000000001, '82.9_Business support service activities n.e.c.': 0.011999999999999999, '78.

 21%|██▏       | 332/1555 [03:56<17:54,  1.14it/s]

...

{'D_ELECTRICITY, GAS, STEAM AND AIR CONDITIONING SUPPLY': 0.01525, 'K_FINANCIAL AND INSURANCE ACTIVITIES': 0.013055555555555555, 'B_MINING AND QUARRYING': 0.012933333333333333, 'E_WATER SUPPLY; SEWERAGE, WASTE MANAGEMENT AND REMEDIATION ACTIVITIES': 0.003777777777777778, 'L_REAL ESTATE ACTIVITIES': 0.0032500000000000003, 'H_TRANSPORTATION AND STORAGE': 0.002826086956521739, 'N_ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES': 0.002575757575757576, 'I_ACCOMMODATION AND FOOD SERVICE ACTIVITIES': 0.002, 'G_WHOLESALE AND RETAIL TRADE; REPAIR OF MOTOR VEHICLES AND MOTORCYCLES': 0.0015934065934065933, 'M_PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIVITIES': 0.0014736842105263158, 'F_CONSTRUCTION': 0.0008636363636363636, 'T_ACTIVITIES OF HOUSEHOLDS AS EMPLOYERS; UNDIFFERENTIATED GOODS- AND SERVICES-PRODUCING ACTIVITIES OF HOUSEHOLDS FOR OWN USE': 0.0006666666666666666, 'O_PUBLIC ADMINISTRATION AND DEFENCE; COMPULSORY SOCIAL SECURITY': 0.0006666666666666666, 'A_AGRICULTURE, FORESTRY AND FISHING': 

 55%|█████▌    | 861/1555 [3:55:43<2:17:47, 11.91s/it]

{'K_FINANCIAL AND INSURANCE ACTIVITIES': 0.021722222222222223, 'D_ELECTRICITY, GAS, STEAM AND AIR CONDITIONING SUPPLY': 0.012624999999999999, 'M_PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIVITIES': 0.005842105263157895, 'N_ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES': 0.004454545454545454, 'L_REAL ESTATE ACTIVITIES': 0.00375, 'I_ACCOMMODATION AND FOOD SERVICE ACTIVITIES': 0.0035, 'T_ACTIVITIES OF HOUSEHOLDS AS EMPLOYERS; UNDIFFERENTIATED GOODS- AND SERVICES-PRODUCING ACTIVITIES OF HOUSEHOLDS FOR OWN USE': 0.0033333333333333335, 'O_PUBLIC ADMINISTRATION AND DEFENCE; COMPULSORY SOCIAL SECURITY': 0.0031111111111111114, 'H_TRANSPORTATION AND STORAGE': 0.0028695652173913043, 'G_WHOLESALE AND RETAIL TRADE; REPAIR OF MOTOR VEHICLES AND MOTORCYCLES': 0.00210989010989011, 'B_MINING AND QUARRYING': 0.0015333333333333334, 'F_CONSTRUCTION': 0.0011363636363636365, 'S_OTHER SERVICE ACTIVITIES': 0.0010526315789473684, 'E_WATER SUPPLY; SEWERAGE, WASTE MANAGEMENT AND REMEDIATION ACTIVITIES': 0.0008888

 55%|█████▌    | 863/1555 [3:56:50<2:53:21, 15.03s/it]2025-11-12 22:11:27.978 python[39526:283530] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-39526-2025-11-12_22_11_27-1943156338‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
 56%|█████▌    | 866/1555 [3:57:58<3:09:20, 16.49s/it]


OSError: [Errno 28] No space left on device: '../results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4/China Maple Leaf Educational Systems Ltd.1.txt/relevant_sentences_China Maple Leaf Educational Systems Ltd.1.txt'

In [ ]:
int("47.1")

ValueError: invalid literal for int() with base 10: '47.1'

In [ ]:
df_nace_codes_descriptions = pd.read_csv("../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
label = 9.1
label = 96.02

In [ ]:
df_nace_codes_descriptions[df_nace_codes_descriptions["CODE"] == str(label)]["NAME"].iloc[0]

'Hairdressing and other beauty treatment'